In [ ]:
import os, re, math, csv, time, zipfile, warnings
from collections import Counter
from zipfile import ZipFile, BadZipFile

import pandas as pd
import openpyxl
import xlrd
import pytesseract
from PIL import Image

# silence noisy warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
warnings.filterwarnings("ignore", message="Palette images with Transparency")

# optional xlsb
supports_xlsb = True
try:
    from pyxlsb import open_workbook as open_xlsb
except Exception:
    supports_xlsb = False
    print("ℹ️ pyxlsb not installed: .xlsb files will be skipped (content sniff will still detect them).")

# ---------- helpers ----------
def calculate_entropy(text: str) -> float:
    if not text:
        return 0.0
    from collections import Counter
    probs = [n / len(text) for n in Counter(text).values()]
    return -sum(p * math.log2(p) for p in probs)

def count_pattern_occurrences(texts, pattern):
    if isinstance(texts, str):
        texts = [texts]
    return sum(len(re.findall(pattern, t)) for t in texts)

def extract_preview_image_text_zip(filepath):
    try:
        with ZipFile(filepath, 'r') as z:
            for name in z.namelist():
                if name.lower().startswith("xl/media") and name.lower().endswith((".jpg", ".jpeg", ".png")):
                    with z.open(name) as img_file:
                        img = Image.open(img_file).convert("RGB")
                        text = pytesseract.image_to_string(img)
                        return text, img.size
    except Exception:
        pass
    return "", (0, 0)

def init_feature_dict(filepath):
    try:
        size = os.path.getsize(filepath)
    except Exception:
        size = 0
    return {
        "file_path": filepath,
        "file_size": size,
        "sheet_count": 0, "max_rows": 0, "max_cols": 0, "total_cells": 0,
        "non_empty_cells": 0, "numeric_cell_count": 0, "string_cell_count": 0,
        "formula_count": 0, "hyperlink_count": 0, "avg_cell_length": 0.0,
        "entropy_of_text": 0.0, "base64_pattern_count": 0, "hex_pattern_count": 0,
        "has_macro": 0, "remote_template_present": 0,
        "ocr_extracted_text_length": 0, "preview_image_text_entropy": 0.0,
        "deceptive_keywords_count_ocr": 0,
        "macro_line_count": 0, "macro_procedure_count": 0, "macro_chr_count": 0,
        "macro_string_function_count": 0, "macro_arithmetic_operator_count": 0,
        "macro_concatenation_count": 0, "macro_callbyname_count": 0,
        "macro_comment_lines": 0, "macro_average_line_length": 0.0,
        "macro_token_count": 0, "macro_count": 0,
        "uses_file_api": 0, "uses_network_api": 0, "uses_process_api": 0,
        "merged_cells_count": 0, "hidden_sheets_count": 0, "protected_sheets_count": 0,
        "named_ranges_count": 0, "empty_sheet_count": 0, "rich_text_formatting_count": 0,
        "macro_count_parentheses": 0, "macro_count_assignments": 0,
        "macro_max_line_length": 0, "macro_max_string_literals": 0,
        "macro_max_arithmetic_ops": 0, "macro_max_concat_ops": 0, "macro_vocab_size": 0,
        "preview_image_width": 0, "preview_image_height": 0,
    }

# ---------- readers ----------
def extract_from_xlsx_xlsm(filepath, features):
    wb = openpyxl.load_workbook(filepath, data_only=False, keep_links=True)
    features["sheet_count"] = len(wb.sheetnames)
    features["named_ranges_count"] = len(list(wb.defined_names))
    all_text = []
    for sheet in wb.worksheets:
        rows = sheet.max_row or 0
        cols = sheet.max_column or 0
        features["max_rows"] = max(features["max_rows"], rows)
        features["max_cols"] = max(features["max_cols"], cols)
        features["total_cells"] += rows * cols
        if sheet.sheet_state != "visible":
            features["hidden_sheets_count"] += 1
        if getattr(sheet, "protection", None) and sheet.protection.sheet:
            features["protected_sheets_count"] += 1
        try:
            if not any(cell.value for row in sheet.iter_rows() for cell in row):
                features["empty_sheet_count"] += 1
        except Exception:
            pass
        try:
            features["merged_cells_count"] += len(sheet.merged_cells.ranges)
        except Exception:
            pass
        for row in sheet.iter_rows():
            for cell in row:
                val = str(cell.value).strip() if cell.value is not None else ''
                if val:
                    features["non_empty_cells"] += 1
                    all_text.append(val)
                    if isinstance(cell.value, str):
                        features["string_cell_count"] += 1
                    elif isinstance(cell.value, (int, float)):
                        features["numeric_cell_count"] += 1
                    if cell.hyperlink:
                        features["hyperlink_count"] += 1
                if cell.data_type == 'f':
                    features["formula_count"] += 1
                if getattr(cell, "font", None) and (cell.font.bold or cell.font.italic or cell.font.underline):
                    features["rich_text_formatting_count"] += 1
    try:
        with ZipFile(filepath, 'r') as z:
            for name in z.namelist():
                if "vbaProject.bin" in name:
                    features["has_macro"] = 1
                    features["macro_count"] += 1
                if name.lower().endswith(".xml"):
                    with z.open(name) as xf:
                        content = xf.read().decode(errors='ignore').lower()
                        if "http://" in content or "https://" in content:
                            features["remote_template_present"] = 1
    except Exception:
        pass
    return all_text

def extract_from_xls(filepath, features):
    wb = xlrd.open_workbook(filepath, formatting_info=False)
    features["sheet_count"] = wb.nsheets
    all_text = []
    for sheet in wb.sheets():
        rows, cols = sheet.nrows, sheet.ncols
        features["max_rows"] = max(features["max_rows"], rows)
        features["max_cols"] = max(features["max_cols"], cols)
        features["total_cells"] += rows * cols
        non_empty = 0
        for r in range(rows):
            for c in range(cols):
                try:
                    val = sheet.cell_value(r, c)
                except Exception:
                    val = ''
                if val not in ('', None):
                    non_empty += 1
                    sval = str(val).strip()
                    all_text.append(sval)
                    if isinstance(val, str):
                        features["string_cell_count"] += 1
                    elif isinstance(val, (int, float)):
                        features["numeric_cell_count"] += 1
        features["non_empty_cells"] += non_empty
    return all_text

def extract_from_xlsb(filepath, features):
    if not supports_xlsb:
        raise RuntimeError("xlsb unsupported (pyxlsb not installed)")
    all_text, sheet_count = [], 0
    with open_xlsb(filepath) as wb:
        for sheet_name in wb.sheets:
            sheet_count += 1
            with wb.get_sheet(sheet_name) as sh:
                for row in sh.rows():
                    for cell in row:
                        v = cell.v
                        if v is not None and str(v).strip() != '':
                            sval = str(v).strip()
                            all_text.append(sval)
                            features["non_empty_cells"] += 1
                            if isinstance(v, str):
                                features["string_cell_count"] += 1
                            elif isinstance(v, (int, float)):
                                features["numeric_cell_count"] += 1
    features["sheet_count"] = sheet_count
    return all_text

# ---------- content-based sniff ----------
def likely_excel(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in {".xlsx", ".xlsm", ".xls"} or (ext == ".xlsb" and supports_xlsb):
        return True
    # Try OpenXML by ZIP structure
    try:
        with zipfile.ZipFile(path) as z:
            return any(n.startswith("xl/") for n in z.namelist())
    except Exception:
        pass
    # Try XLSB magic (D0 CF 11 E0 for old CFB? modern xlsb is ZIP? Many xlsb are CFB streams)
    try:
        with open(path, "rb") as f:
            header = f.read(8)
        # CFBF header for some Office binaries
        if header.startswith(b'\xD0\xCF\x11\xE0'):
            # could be xls, doc, ppt; we still let our per-format open decide later
            return True
    except Exception:
        pass
    return False

# ---------- unified extraction ----------
def extract_excel_features(filepath):
    features = init_feature_dict(filepath)
    all_text = []
    ext = os.path.splitext(filepath)[1].lower()
    try:
        if ext in (".xlsx", ".xlsm") or _is_openxml_zip(filepath):
            all_text = extract_from_xlsx_xlsm(filepath, features)
        elif ext == ".xls":
            all_text = extract_from_xls(filepath, features)
        elif ext == ".xlsb":
            all_text = extract_from_xlsb(filepath, features)
        else:
            # fallback heuristics
            try:
                with ZipFile(filepath, 'r') as _:
                    all_text = extract_from_xlsx_xlsm(filepath, features)
            except Exception:
                all_text = extract_from_xls(filepath, features)  # may still fail; caught below

        # common post
        features["avg_cell_length"] = (sum(len(t) for t in all_text) / len(all_text)) if all_text else 0.0
        joined = "\n".join(all_text)
        features["entropy_of_text"] = calculate_entropy(joined)
        features["base64_pattern_count"] = count_pattern_occurrences(
            all_text, r'(?:[A-Za-z0-9+/]{4}){10,}(?:[A-Za-z0-9+/]{2}==|[A-Za-z0-9+/]{3}=)?'
        )
        features["hex_pattern_count"] = count_pattern_occurrences(
            all_text, r'(?:\\x[0-9a-fA-F]{2}){4,}'
        )
        lines = joined.splitlines()
        tokens = re.findall(r'\w+', joined)
        features["macro_line_count"] = len(lines)
        features["macro_procedure_count"] = sum(1 for l in lines if re.search(r'\b(Sub|Function)\b', l, re.IGNORECASE))
        features["macro_chr_count"] = joined.lower().count("chr")
        features["macro_string_function_count"] = sum(joined.lower().count(f) for f in ["replace", "ucase", "lcase", "split", "instr", "strreverse"])
        features["macro_arithmetic_operator_count"] = sum(joined.count(op) for op in ["+", "-", "*", "/"])
        features["macro_concatenation_count"] = joined.count("&")
        features["macro_callbyname_count"] = joined.lower().count("callbyname")
        features["macro_comment_lines"] = sum(1 for l in lines if l.strip().startswith("'"))
        features["macro_average_line_length"] = (sum(len(l) for l in lines) / len(lines)) if lines else 0.0
        features["macro_token_count"] = len(set(tokens))
        features["macro_count_parentheses"] = joined.count("(") + joined.count(")")
        features["macro_count_assignments"] = joined.count("=")
        features["macro_max_line_length"] = max((len(l) for l in lines), default=0)
        features["macro_max_string_literals"] = max((l.count('"') for l in lines), default=0)
        features["macro_max_arithmetic_ops"] = max((sum(l.count(op) for op in "+-*/") for l in lines), default=0)
        features["macro_max_concat_ops"] = max((l.count("&") for l in lines), default=0)
        features["macro_vocab_size"] = len(set(tokens))

        features["uses_file_api"] = int(any(k in joined for k in ['Open', 'Write', 'SaveAs', 'FileCopy']))
        features["uses_network_api"] = int(any(k in joined for k in ['XMLHttpRequest', 'WinHttpRequest', 'URLDownloadToFile']))
        features["uses_process_api"] = int(any(k in joined for k in ['Shell', 'CreateProcess', 'Wscript.Shell']))

        ocr_text, (w, h) = extract_preview_image_text_zip(filepath)
        features["ocr_extracted_text_length"] = len(ocr_text)
        features["preview_image_text_entropy"] = calculate_entropy(ocr_text)
        features["preview_image_width"] = w
        features["preview_image_height"] = h
        suspicious_phrases = ["enable content", "click here", "view document", "macro", "enable editing"]
        features["deceptive_keywords_count_ocr"] = sum(ocr_text.lower().count(p) for p in suspicious_phrases)

    except Exception as e:
        raise e
    return features

def _is_openxml_zip(path):
    try:
        with ZipFile(path) as z:
            return any(n.startswith("xl/") for n in z.namelist())
    except Exception:
        return False

# ---------- robust batch w/ diagnostics ----------
def process_folder_debug(input_folder, output_excel_path, errors_csv_path=None, sample_print=20):
    if not os.path.exists(input_folder):
        raise FileNotFoundError(f"Input folder does not exist: {input_folder}")
    if not os.path.isdir(input_folder):
        raise NotADirectoryError(f"Input path is not a directory: {input_folder}")

    # quick scan
    top_items = os.listdir(input_folder)
    print(f"Top-level items in input: {len(top_items)} (showing up to {min(sample_print,len(top_items))})")
    for x in top_items[:sample_print]:
        print(" •", x)

    dirs_seen = files_seen = 0
    exts = Counter()
    sample_paths = []
    for root, dirs, files in os.walk(input_folder):
        dirs_seen += len(dirs)
        files_seen += len(files)
        for f in files:
            exts[os.path.splitext(f)[1].lower()] += 1
            if len(sample_paths) < sample_print:
                fp = os.path.join(root, f)
                try:
                    sz = os.path.getsize(fp)
                except Exception:
                    sz = -1
                sample_paths.append((fp, sz))
    print(f"Dirs seen: {dirs_seen} | Files seen: {files_seen}")
    print("Extension histogram:", exts)
    print("Sample files (path, size bytes):")
    for p, s in sample_paths:
        print(" -", p, s)

    results = []
    total = 0
    matched = 0
    errors = []
    ordered_cols = None

    # process
    for root, _, files in os.walk(input_folder):
        for fname in files:
            fpath = os.path.join(root, fname)
            total += 1
            try:
                if likely_excel(fpath):
                    matched += 1
                    feats = extract_excel_features(fpath)
                    if ordered_cols is None:
                        ordered_cols = list(feats.keys())
                    results.append(feats)
            except Exception as e:
                errors.append([fpath, type(e).__name__, str(e)[:400]])

            # incremental write every 2000 files or last chunk
            if matched % 2000 == 0 and matched > 0:
                df = pd.DataFrame(results)
                if ordered_cols:
                    df = df.reindex(columns=ordered_cols)
                os.makedirs(os.path.dirname(output_excel_path), exist_ok=True)
                df.to_excel(output_excel_path, index=False, engine="openpyxl")
                print(f"💾 checkpoint: wrote {len(df)} rows so far...")

    # final write
    df = pd.DataFrame(results)
    if ordered_cols:
        df = df.reindex(columns=ordered_cols)
    os.makedirs(os.path.dirname(output_excel_path), exist_ok=True)
    df.to_excel(output_excel_path, index=False, engine="openpyxl")

    # errors log
    if errors_csv_path:
        with open(errors_csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["file_path", "error_type", "message"])
            w.writerows(errors)
        print(f"🧾 Wrote errors log: {errors_csv_path} (rows: {len(errors)})")

    print(f"✅ Extracted features from {len(df)} files and saved to {output_excel_path}")
    print(f"🔍 Scanned: {total} files | Matched (likely excel): {matched} | Skipped: {total - matched}")



input_folder = "Put your input path here"
output_excel = "Put your output path here"
errors_csv = "A pathway for tracking the corrupted files"

process_folder_debug(input_folder, output_excel, errors_csv_path=errors_csv, sample_print=25)
